[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [PyMongo and Beanie, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pymongo-and-beanie-deep-dive.html)

# Collections and Documents &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell is the notebook's boot cell, which starts MongoDB and seeds it. Run it first. Each
task opens its own client, drops what it uses, and closes, so they can be run in any order.


In [1]:
import os
import random
import subprocess
import sys
import time
from importlib.metadata import PackageNotFoundError, version

try:
    if version("pymongo") != "4.18.1" or version("beanie") != "2.2.0":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "pymongo==4.18.1", "beanie==2.2.0"], check=True)

import pymongo
from bson import ObjectId

DBPATH = "/content/mongo" if os.path.isdir("/content") else "/tmp/guide_mongo/rs"
LOGPATH = f"{DBPATH}.log"
URI = "mongodb://127.0.0.1:27017/shop"                              # no credential, anywhere
PUBLISHED = ["jammy", "noble"]                                      # codenames MongoDB builds for


def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(timeout=2000):
    """Whether a mongod is there, asked directly rather than through topology discovery."""
    try:
        with pymongo.MongoClient("mongodb://127.0.0.1:27017/?directConnection=true",
                                 serverSelectionTimeoutMS=timeout) as client:
            client.admin.command("ping")
            return True
    except pymongo.errors.PyMongoError:
        return False


def install_server():
    """Add MongoDB's own apt repository and install the server package. Linux only."""
    if shell("which mongod")[0] == 0:
        return "already installed"

    codename = shell("lsb_release -cs")[1]
    if codename not in PUBLISHED:                                   # an unpublished one breaks apt
        print(f"  Ubuntu '{codename}' has no MongoDB repository; using '{PUBLISHED[-1]}' instead")
        codename = PUBLISHED[-1]

    if not shell("grep -o avx /proc/cpuinfo | head -1")[1]:
        raise RuntimeError("This CPU has no AVX. Every MongoDB build since 5.0 needs it, so "
                           "neither the apt package nor the tarball will start here.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"curl -fsSL https://www.mongodb.org/static/pgp/server-8.0.asc "
          f"| {sudo}gpg --dearmor -o /usr/share/keyrings/mongodb-8.0.gpg")
    shell(f'echo "deb [signed-by=/usr/share/keyrings/mongodb-8.0.gpg] '
          f'https://repo.mongodb.org/apt/ubuntu {codename}/mongodb-org/8.0 multiverse" '
          f'| {sudo}tee /etc/apt/sources.list.d/mongodb-8.0.list')
    shell(f"{sudo}apt-get -qq update "                              # this one list file only
          f"-o Dir::Etc::sourcelist=sources.list.d/mongodb-8.0.list "
          f"-o Dir::Etc::sourceparts=-")
    code, out = shell(f"{sudo}apt-get -qq -y install mongodb-org-server")
    if shell("which mongod")[0] != 0:
        raise RuntimeError(f"mongodb-org-server did not install. apt said: {out[-400:]}")
    return f"installed from the {codename} repository"


def start_server(wait=30):
    """Start mongod with a replica set name, idempotently. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No mongod is answering on 127.0.0.1:27017. Start your own server "
                           "with --replSet rs0 and run this again: this cell only installs one "
                           "on Linux, which is what Colab runs.")

    print(" ", install_server())
    os.makedirs(DBPATH, exist_ok=True)
    code, out = shell(f"mongod --dbpath {DBPATH} --replSet rs0 --bind_ip 127.0.0.1 "
                      f"--fork --logpath {LOGPATH}")
    if code != 0:                                                   # --fork hides the reason
        print("  mongod did not start. The last lines of its log:")
        print("   ", shell(f"tail -20 {LOGPATH}")[1].replace("\n", "\n    "))
        raise RuntimeError("mongod exited. The log above says why.")

    for attempt in range(1, wait + 1):
        if answering():
            return "installed and started"
        print(f"  waiting for mongod ({attempt})")
        time.sleep(1)
    raise RuntimeError(f"mongod did not answer within {wait} seconds.")

def initiate(wait=30):
    """Make the single node a replica set, which is what transactions and migrations need."""
    with pymongo.MongoClient("mongodb://127.0.0.1:27017/?directConnection=true",
                             serverSelectionTimeoutMS=2000) as boot:
        try:                                                        # an explicit host, not getHostName()
            boot.admin.command("replSetInitiate",
                               {"_id": "rs0", "members": [{"_id": 0, "host": "127.0.0.1:27017"}]})
        except pymongo.errors.OperationFailure as error:
            if error.code != 23:                                    # 23 is AlreadyInitialized
                raise

        for attempt in range(1, wait + 1):
            hello = boot.admin.command("hello")
            if hello.get("isWritablePrimary"):
                return f"replica set {hello['setName']}, primary"
            time.sleep(1)
    raise RuntimeError(f"No primary after {wait} seconds. The last hello was: {hello}")

SIZE = 500                                                          # Indexes and the catalog raise this
KINDS = ["laptop", "monitor", "keyboard", "mouse", "cable"]
MAKERS = ["Aster", "Belden", "Corvid", "Dalgo"]


def seed(size=None, force=False):
    """Fill shop.products and shop.reviews, once, from a fixed seed so every run agrees."""
    size = SIZE if size is None else size
    client = pymongo.MongoClient(URI, tz_aware=True)
    shop = client.get_default_database()

    if not force and shop.products.estimated_document_count() == size:
        client.close()
        return size

    shop.products.drop()
    shop.reviews.drop()
    random.seed(0)                                                  # the whole reason runs agree

    products, reviews = [], []
    for number in range(size):
        kind = KINDS[number % len(KINDS)]
        product = {
            "_id": number,
            "sku": f"{kind[:3].upper()}-{number:06d}",
            "name": f"{MAKERS[number % len(MAKERS)]} {kind} {number}",
            "maker": MAKERS[number % len(MAKERS)],
            "kind": kind,
            "price": round(random.uniform(5, 2000), 2),
            "stock": random.randint(0, 400),
            "tags": sorted(random.sample(["sale", "new", "refurbished", "bulk", "clearance"], 2)),
            "size": {"w": random.randint(5, 60), "h": random.randint(2, 40)},
        }
        products.append(product)
        for _ in range(random.randint(0, 3)):
            reviews.append({"product_id": number, "stars": random.randint(1, 5),
                            "body": f"A review of {product['name']}"})

    for start in range(0, len(products), 5000):                     # batches, not one huge insert
        shop.products.insert_many(products[start:start + 5000])
    for start in range(0, len(reviews), 5000):
        shop.reviews.insert_many(reviews[start:start + 5000])

    client.close()
    return size


def report():
    """One line naming what this notebook is running against."""
    with pymongo.MongoClient(URI, tz_aware=True) as client:
        build = client.admin.command("buildInfo")["version"].split(".")[0]
        shop = client.get_default_database()
        return (f"MongoDB {build} | pymongo {version('pymongo')} | beanie {version('beanie')} "
                f"| products: {shop.products.count_documents({})}")


print("server: ", start_server())
print("replica:", initiate())
print("seeded: ", seed(), "products")
print(report())


server:  already running
replica: replica set rs0, primary
seeded:  500 products
MongoDB 8 | pymongo 4.18.1 | beanie 2.2.0 | products: 500


**1.** One insert, and the receipt.


In [2]:
client = pymongo.MongoClient(URI, tz_aware=True)
shop = client.get_default_database()
shop.answers.drop()

result = shop.answers.insert_one({"task": 1, "done": True})

print("type:        ", type(result).__name__)
print("acknowledged:", result.acknowledged)
print("id is an:    ", type(result.inserted_id).__name__)
client.close()


type:         InsertOneResult
acknowledged: True
id is an:     ObjectId


The id itself is not printed, because it is twelve mostly random bytes and would be different every
time this notebook runs. Its type is the part that is worth knowing.


**2.** The dictionary, before and after.


In [3]:
client = pymongo.MongoClient(URI, tz_aware=True)
shop = client.get_default_database()
shop.answers.drop()

document = {"task": 2}
print("before:", sorted(document))
shop.answers.insert_one(document)
print("after: ", sorted(document))
print("it is the same object, not a copy:", "_id" in document)
client.close()


before: ['task']
after:  ['_id', 'task']
it is the same object, not a copy: True


PyMongo generates the `_id` on this side of the wire and writes it into the dictionary it was
given. Handing the same dictionary to a second `insert_one` is therefore a duplicate key error.


**3.** Five in one call.


In [4]:
client = pymongo.MongoClient(URI, tz_aware=True)
shop = client.get_default_database()
shop.answers.drop()

result = shop.answers.insert_many([{"n": number} for number in range(5)])

print("ids returned:", len(result.inserted_ids))
print("documents:   ", shop.answers.count_documents({}))
client.close()


ids returned: 5
documents:    5


One round trip rather than five. On a local socket the difference is small; against a server across
a network it is the difference between one wait and five.


**4.** An _id of your own, twice.


In [5]:
client = pymongo.MongoClient(URI, tz_aware=True)
shop = client.get_default_database()
shop.answers.drop()

shop.answers.insert_one({"_id": "only-one", "attempt": 1})
try:
    shop.answers.insert_one({"_id": "only-one", "attempt": 2})
except pymongo.errors.DuplicateKeyError as error:
    print("second attempt:", type(error).__name__)

print("what is stored:", shop.answers.find_one({"_id": "only-one"}))
client.close()


second attempt: DuplicateKeyError
what is stored: {'_id': 'only-one', 'attempt': 1}


The first write stands and the second is refused. That is what makes a chosen `_id` the simplest
way to keep a loader from writing anything twice.


**5.** A collection that is not there.


In [6]:
client = pymongo.MongoClient(URI, tz_aware=True)
shop = client.get_default_database()

print("count:  ", shop.not_a_collection.count_documents({}))
print("exists: ", "not_a_collection" in shop.list_collection_names())
print("find_one:", shop.not_a_collection.find_one({}))
client.close()


count:   0
exists:  False
find_one: None


Zero, `False` and `None`. No exception anywhere, which is why a query that mysteriously returns
nothing is worth checking against `list_collection_names()` before anything else.


**6.** The exact count and the estimate.


In [7]:
client = pymongo.MongoClient(URI, tz_aware=True)
shop = client.get_default_database()

exact = shop.products.count_documents({})
estimate = shop.products.estimated_document_count()

print("count_documents({}):       ", exact)
print("estimated_document_count():", estimate)
print("they agree here:", exact == estimate)
client.close()


count_documents({}):        500
estimated_document_count(): 500
they agree here: True


They agree because nothing has crashed and no write is in flight. The estimate comes from collection
metadata rather than from counting, so it is instant and it cannot take a filter, and after an
unclean shutdown it can be wrong until the collection is scanned again.


---

&#8592; **Back to:** [Collections and Documents](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pymongo-and-beanie-deep-dive/02-collections-and-documents.ipynb)  &nbsp;&middot;&nbsp;  [PyMongo and Beanie, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pymongo-and-beanie-deep-dive.html)
